# Out-of-fold models · Folds 2 & 3

**EdgeAttNet / DINOv3 ConvNeXt-Base** · Training-only cross-fitting

The four OOF models supply held-out probability maps for the learned component filter. This notebook handles **folds 2 & 3**. It is not the final competition prediction model, and these fold weights are not an ensemble at submission time.

| Mode | Effect |
|:--|:--|
| **Pretrained · default** | Load the trained fold EMA models, verify their original fold membership, then reuse or regenerate their held-out probability maps |
| **Train from source** | Fine-tune the original DINOv3-Base encoder for 55 epochs in each fold, save final EMA, and infer on the excluded training images |

**Data integrity.** Four temporal GroupKFold folds · 2-day embargo · seed 42 · no use of the 141 validation images for fold training. Each held-out image is predicted by a model that never trained on it.

In [ ]:
# Locate the repository from its root or notebooks/ directory.
from pathlib import Path
import sys
REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p/'src'/'architecture.py').is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Open this notebook inside the extracted repository.')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
import os
os.chdir(REPO_ROOT)


## 01 / Execution settings

The default path searches attached OOF artifacts first, then downloads the **four-fold model-only EMA weights; companion JSON metadata is generated automatically from the bundled original manifests** from the public Hugging Face repository when needed. Each downloaded pair is verified against the reconstructed training-fold manifest before inference. Fresh training remains optional and requires the DINOv3 encoder initialization.

In [ ]:
from pathlib import Path
import subprocess, sys, json, os

TRAIN_OOF_FROM_SCRATCH = False   # Default: load the pretrained fold models
REUSE_PRECOMPUTED_OOF_MAPS = True
EXPORT_MODEL_ONLY = True        # Effective only after a completed new training run

FOLD_PAIR = '23'
REPO_ROOT = Path.cwd()
DATA_ROOT = None                # Optional explicit competition data root
SPLIT_ROOT = REPO_ROOT/'data'/'splits'  # Curated split bundled with source
OOF_MODELS_ROOT = Path('/kaggle/input')  # Prefer attached fold weights; otherwise use HF
HF_OOF_REPO = 'youssefElghanam191/edgeattnet-dinov3-convnext-base'
DOWNLOAD_OOF_FROM_HF = True
PRETRAINED_DINO_ENCODER = None  # Required ONLY when TRAIN_OOF_FROM_SCRATCH=True
RESUME_ROOT = None              # Prior complete/partial pair output for training restart
SESSION_HOURS = 9.5
OUTPUT_ROOT = Path('/kaggle/working/DINO_OOF_Folds23') if Path('/kaggle/working').exists() else REPO_ROOT/'outputs'/'DINO_OOF_Folds23'

assert (REPO_ROOT/'scripts'/'run_oof_pair.py').is_file(), (
    'Open the notebook from the repository root, not as a detached Kaggle notebook. '
    'It needs src/oof_training_worker.py and scripts/run_oof_pair.py.')
print('Fold pair:',FOLD_PAIR,'| mode:', 'TRAIN' if TRAIN_OOF_FROM_SCRATCH else 'PRETRAINED')

## 02 / Fold training or pretrained inference

The runner preserves the original fixed-epoch training recipe, temporal fold manifest, native-resolution TTA8, and FP16 probability-cache format. Training uses two GPUs. Pretrained inference accepts one GPU and runs the models sequentially if necessary. An interrupted training session can be resumed by attaching the *entire* prior output and setting `RESUME_ROOT`.

In [ ]:
command = [sys.executable, str(REPO_ROOT/'scripts'/'run_oof_pair.py'),
           '--folds',FOLD_PAIR,
           '--mode','train' if TRAIN_OOF_FROM_SCRATCH else 'pretrained',
           '--models-root',str(OOF_MODELS_ROOT),
           '--output',str(OUTPUT_ROOT),'--hours',str(SESSION_HOURS),
           '--hf-repo',HF_OOF_REPO]
if not DOWNLOAD_OOF_FROM_HF:command.append('--no-hf-download')
if not REUSE_PRECOMPUTED_OOF_MAPS:command.append('--no-reuse-maps')
if DATA_ROOT is not None:command += ['--data-root',str(DATA_ROOT)]
if SPLIT_ROOT is not None:command += ['--splits-root',str(SPLIT_ROOT)]
if RESUME_ROOT is not None:command += ['--resume-root',str(RESUME_ROOT)]
if TRAIN_OOF_FROM_SCRATCH:
    if PRETRAINED_DINO_ENCODER is None:
        raise ValueError('Fresh fold training requires PRETRAINED_DINO_ENCODER. '
                         'This is the official DINOv3 ConvNeXt-Base encoder initialization.')
    command += ['--encoder',str(PRETRAINED_DINO_ENCODER)]
subprocess.run(command,cwd=str(REPO_ROOT),check=True)

## 03 / Verify produced artifacts

Each OOF map has a SHA-256-checked sidecar identifying the excluded training image, fold, model, and original split manifest. All held-out images for this pair must be present before results are considered complete.

In [ ]:
status_path=OUTPUT_ROOT/'pair_status.json'
assert status_path.is_file(),f'OOF process produced no status report: {status_path}'
status=json.loads(status_path.read_text())
assert status['complete'] and len(status['folds'])==2 and all(x['complete'] for x in status['folds'])
for item in status['folds']:
    print(f"Fold {item['fold']}: {item['heldout_images']} excluded-image probability maps verified")
print('Manifest:',OUTPUT_ROOT/'oof_training_manifest.json')
print('OOF maps:',OUTPUT_ROOT/'prob_oof')

## 04 / Publish model-only EMA weights (optional)

After fresh training, the fold's `fixed_final_ema.pth` contains a training wrapper. For deployment, export **just the EMA state dictionary** and keep its small fold-provenance JSON alongside it. No optimizer or intermediate training state is included in the published model.

In [ ]:
if TRAIN_OOF_FROM_SCRATCH and EXPORT_MODEL_ONLY:
    manifest=OUTPUT_ROOT/'oof_training_manifest.json'
    for fold in (int(x) for x in FOLD_PAIR):
        subprocess.run([sys.executable,str(REPO_ROOT/'scripts'/'export_oof_models.py'),
                        '--source',str(OUTPUT_ROOT/f'fold_{fold}'/'fixed_final_ema.pth'),
                        '--out',str(OUTPUT_ROOT/'models'/'oof'),'--manifest',str(manifest)],
                       cwd=str(REPO_ROOT),check=True)
    print('Exported model-only OOF weights to:',OUTPUT_ROOT/'models'/'oof')
else:
    print('Model-only export skipped; fold inference artifacts are available.')

---

**Next stage.** After *both* pair notebooks are complete, run [`02_Filter_Training.ipynb`](02_Filter_Training.ipynb) to audit coverage of all **566** training images and optionally retrain the CatBoost selector. The final submission demo does **not** need the OOF models when using the published CatBoost artifact.